<a href="https://colab.research.google.com/github/mgd-code/llm-workshop-witten/blob/main/performance_evaluation_metrics_git.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Performance evaluation of large language models

In this script, we use the scikit-learn package to evaluate model performance.
Since we don't have our own validation sample to compare, we use simulated data.

## 1. Set up

Here we load the scikit-learn package

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## 2. Simulated data (binary classification)

Here we simulate some data, as if our classification scheme only had two categories: 0 and 1. For example, this could be the case if the LLM's task was to classify excerpts as about climate change (1) or not about climate change (0).

"Truth" here would be the gold-standard, human-coded validation sample. "LLM" would be the LLM outputs.

In [ ]:
# The TRUE answers for 10 texts
truth = [1, 1, 1, 1, 0, 0, 0, 0, 1, 0]

# What the LLM answered for the same 10 texts
llm   = [1, 1, 1, 0, 0, 1, 1, 0, 1, 0]

## 3. Performance metrics

Now we will use the functions we imported from scikit-learn to calculate the accuracy, precision, recall, and F1 scores.

**Accuracy** = what fraction of answers were correct overall

**Precision** = sensitive to false positives. When the model gives a positive answer, how likely is it to be correct?

**Recall** = sensitive to false negatives. Of all of the true positives, how many did the model actually catch?

**F1 score** = balance of precision and recall

Our natural incliniation is often to think of accuracy first. However, accuracy has some big limitations, especially in cases where the sample is imbalanced. So, for example, if it's the case that 95 out of 100 central banking excerpts aren't about climate change, but 5 are, the model can get a really high accuracy score by simply predicted negative in every case. However, what we really care about is picking up the positives. That's where precision, recall, and F1 scores come in.

### The metrics in plain English

$$\text{Accuracy} = \frac{\text{Number of correct responses}}{\text{Total responses}}$$

$$\text{Precision} = \frac{\text{Correct positive predictions}}{\text{All positive predictions}}$$

$$\text{Recall} = \frac{\text{Correct positive predictions}}{\text{All actual positives}}$$

$$\text{F1} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$


### The metrics in true vs false positives

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

$$\text{Precision} = \frac{TP}{TP + FP}$$

$$\text{Recall} = \frac{TP}{TP + FN}$$

$$\text{F1} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

where **TP** = true positives, **TN** = true negatives, **FP** = false positives, **FN** = false negatives.

## 4. Calculate the metrics (binary example)

Using scikit-learn to do the work for us:

In [ ]:


print("Accuracy: ", accuracy_score(truth, llm))
print("Precision:", precision_score(truth, llm))
print("Recall:   ", recall_score(truth, llm))
print("F1:       ", f1_score(truth, llm))

Let's calculate manually now for comparison. Here's our sample again:

tru = [1, 1, 1, 1, 0, 0, 0, 0, 1, 0]

llm  = [1, 1, 1, 0, 0, 1, 1, 0, 1, 0]

So, we can see that the LLM was right 7 out of 10 times. So, $$ Accuracy = 7 / 10 = 0.7$$

Now let's look at precision. There are four true positives (TP) in our sample: when the LLM predicted 1, and the true answer also was 1. There are also two false positives (FP), where the model predicted 1, but the true answer was 0. So $$ Precision = 4 / (4+2) = 0.67 $$

Next, recall. We know there are four true positives. There is also one false negative, where the model predicted 0 but the true answer was 1. So $$ Recall = 4 / (4 + 1) = 0.8 $$

Finally, F1 is simply the harmonic mean of these two scores. So $$ F1 = 2 * (0.67 * 0.8) / (0.67 + 0.8) = 0.73 $$

## 5. Interpret the metrics (binary)

What does it mean that our recall score is higher than our precision?

It means that our LLM is doing a better job at not missing any true positives (recall), than it is at precisely distinguishing between true positives and false positives.

Our F1 score, at 0.73, is passable (0.7 is sometimes thought of as a floor, though in reality in really depends on the task and F1 scores can often be much higher).

## 6. Simulated data (multi-classification)

Now we simulate some data to represent the case where we have three categories in our scheme: 0, 1, and 2. This more closely resembles the test prompt we used for our climate speeches.

In [ ]:
# The TRUE answers for 10 texts
truth = [0, 0, 0, 0, 1, 1, 1, 2, 2, 2]

# What the LLM answered for the same 10 texts
llm   = [0, 0, 1, 0, 1, 1, 2, 2, 2, 2]

## 7. Calculate performance metrics (multi-classification)

When we have more than two labels, we can still calculate precision, recall, and F1 scores, but we need to modify the logic.

For each label class (e.g., 1), we essentially treat that label as positive, and any other labels (e.g., 0, 2) as negative. Then, we average performance across classes to get our scores.

There are two ways to do this: **macro** or **weighted** average scores. Macro average treats every class the same and takes a simple average, while weighted weights performance by the frequency of each label. So, since 0 is the most common answer in our "truth", the precision and recall scores for 0 would be weighted more heavily than the scores for 1 and 2 in the average.

The command is very similar, except for an addition: average = "macro".

In [ ]:
print("Accuracy: ", accuracy_score(truth, llm))
print("Precision:", precision_score(truth, llm, average="macro"))
print("Recall:   ", recall_score(truth, llm, average="macro"))
print("F1:       ", f1_score(truth, llm, average="macro"))

We could instead calculate weighted scores and compare:

In [ ]:
print("Accuracy: ", accuracy_score(truth, llm))
print("Precision:", precision_score(truth, llm, average="weighted"))
print("Recall:   ", recall_score(truth, llm, average="weighted"))
print("F1:       ", f1_score(truth, llm, average="weighted"))